In [ ]:
### notebooks/06_explainability.ipynb

# Cell 1: Setup
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from src.config import MODELS_DIR, PROCESSED_DATA_DIR
from src.utils.file_handler import load_csv, load_model

sns.set_style("whitegrid")
shap.initjs()

# Cell 2: Load Data and Model
print("Loading data and model...")

train_data = load_csv(PROCESSED_DATA_DIR / "train_test_split" / "train.csv")
test_data = load_csv(PROCESSED_DATA_DIR / "train_test_split" / "test.csv")

X_train = train_data.drop('pass_fail', axis=1)
X_test = test_data.drop('pass_fail', axis=1)
y_test = test_data['pass_fail']

# Load Random Forest (best model typically)
rf_model = load_model(MODELS_DIR / "improved" / "random_forest.pkl")

print("Data and model loaded!")

# Cell 3: Feature Importance (Built-in)
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 20 Features by Importance:")
print(feature_importance.head(20))

# Plot
plt.figure(figsize=(10, 10))
top_20 = feature_importance.head(20)
plt.barh(range(len(top_20)), top_20['importance'], color='steelblue')
plt.yticks(range(len(top_20)), top_20['feature'])
plt.xlabel('Importance', fontsize=12, fontweight='bold')
plt.ylabel('Feature', fontsize=12, fontweight='bold')
plt.title('Top 20 Feature Importance', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Cell 4: SHAP Explainer
print("Creating SHAP explainer...")
print("This may take a few minutes...")

explainer = shap.TreeExplainer(rf_model)

# Use sample for faster computation
X_sample = X_test.sample(min(100, len(X_test)), random_state=42)

shap_values = explainer.shap_values(X_sample)

# For binary classification, use positive class
if isinstance(shap_values, list):
    shap_values_plot = shap_values[1]
else:
    shap_values_plot = shap_values

print("SHAP values calculated!")

# Cell 5: SHAP Summary Plot
print("Creating SHAP summary plot...")

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values_plot, X_sample, show=False)
plt.tight_layout()
plt.show()

# Cell 6: SHAP Bar Plot
print("Creating SHAP importance bar plot...")

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values_plot, X_sample, plot_type="bar", show=False)
plt.tight_layout()
plt.show()

# Cell 7: SHAP Feature Importance
shap_importance = pd.DataFrame({
    'feature': X_sample.columns,
    'shap_importance': np.abs(shap_values_plot).mean(axis=0)
}).sort_values('shap_importance', ascending=False)

print("Top 15 Features by SHAP Importance:")
print(shap_importance.head(15))

# Cell 8: Individual Prediction Explanation
# Pick a student to explain
student_idx = 0
student_data = X_sample.iloc[student_idx:student_idx+1]

print(f"Explaining prediction for student {student_idx}...")
print(f"Predicted probability of passing: {rf_model.predict_proba(student_data)[0][1]:.3f}")

shap.force_plot(
    explainer.expected_value[1], 
    shap_values_plot[student_idx],
    student_data,
    matplotlib=True,
    show=False
)
plt.tight_layout()
plt.show()

# Cell 9: Dependence Plots
# Plot for top 3 features
top_features = shap_importance.head(3)['feature'].tolist()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, feature in enumerate(top_features):
    shap.dependence_plot(
        feature, 
        shap_values_plot, 
        X_sample,
        ax=axes[idx],
        show=False
    )

plt.tight_layout()
plt.show()

# Cell 10: Waterfall Plot
print("Creating waterfall plot for first student...")

shap.waterfall_plot(
    shap.Explanation(
        values=shap_values_plot[0],
        base_values=explainer.expected_value[1],
        data=X_sample.iloc[0],
        feature_names=X_sample.columns.tolist()
    ),
    max_display=15,
    show=False
)
plt.tight_layout()
plt.show()

# Cell 11: Summary
print("\n" + "="*60)
print("EXPLAINABILITY SUMMARY")
print("="*60)
print(f"Model: Random Forest")
print(f"Samples analyzed: {len(X_sample)}")
print(f"\nTop 5 Most Important Features:")
for i, row in feature_importance.head(5).iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")

print(f"\nTop 5 SHAP Important Features:")
for i, row in shap_importance.head(5).iterrows():
    print(f"  {row['feature']}: {row['shap_importance']:.4f}")